# Submission: sub9_convnext_x2
10 models: v6convnext (seed 42) + v6convnext2 (seed 43), both ConvNeXt-T + token head. Tests whether a second seed adds on top of sub6 (public 0.898).

Inputs to attach: the competition; `knee-mri-pipeline` (knee_pipeline_v6, patched model.py); and the committed OUTPUT of two notebooks:
the first ConvNeXt 5-fold notebook (fold0..4_v6convnext.pt) and the second-seed notebook (fold0..4_v6convnext2.pt). Files are found by name, not by notebook slug.
GPU T4 x2, internet OFF. Every number printed under TIMING is measured on this run.

Rank average over every model, so each seed gets equal weight (5 models each).

In [1]:
import os, re, sys, time, json, shutil, subprocess
from pathlib import Path
import numpy as np, pandas as pd
T0 = time.time()
INCLUDE = ['v6convnext', 'v6convnext2']      # groups: base | v4mix | v6tok | v6both | v6dinov2 | v6dino16 | v6convnext | v6convnext2 | ...
BASE_HINT = "knee-mri-diagnosis-training"       # the baseline's plain fold0..4.pt are recognised by this part of their path
N_TEST_FULL_EST = 1300
ROOT = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
W = Path(os.environ.get("KNEE_WORK", "/kaggle/working")); W.mkdir(parents=True, exist_ok=True)
SKIP = {"train_series", "test_series", "cache", "__pycache__", ".git"}
def walk(root, maxdepth=6):
    q = [(Path(root), 0)]
    while q:
        d, k = q.pop(0)
        try: ents = sorted(os.scandir(d), key=lambda e: e.name)
        except OSError: continue
        for e in ents:
            if e.is_dir(follow_symlinks=False):
                if e.name not in SKIP and k < maxdepth: q.append((Path(e.path), k + 1))
            else: yield Path(e.path)
files = list(walk(ROOT)); print("input files scanned:", len(files))

code_dirs = sorted({f.parent for f in files if f.name == "infer.py" and (f.parent / "model.py").exists() and (f.parent / "geom.py").exists()})
assert len(code_dirs) == 1, f"need exactly one knee_pipeline_v6 code folder, found {code_dirs}"
CODE = code_dirs[0]
assert "vit_size" in (CODE / "model.py").read_text() and "canon" in (CODE / "infer.py").read_text(), "code folder is not the latest v6"
data_dirs = sorted({f.parent for f in files if f.name == "test_series.csv" and (f.parent / "sample_submission.csv").exists() and (f.parent / "test.csv").exists()})
assert len(data_dirs) == 1, f"competition data folder: {data_dirs}"
DATA = data_dirs[0]
print("code:", CODE, "\ndata:", DATA)

GROUPS = {}
for tag in INCLUDE:
    pat = r"fold(\d)\.pt" if tag == "base" else rf"fold(\d)_{tag}\.pt"
    found = {}
    for f in files:
        m = re.fullmatch(pat, f.name)
        if not m or (tag == "base" and BASE_HINT not in str(f)): continue
        i = int(m.group(1))
        assert i not in found, f"{tag} fold {i} appears twice: {found[i]} and {f}"
        found[i] = f
    assert sorted(found) == [0, 1, 2, 3, 4], f"group {tag}: folds found {sorted(found)} (attach the notebook output that holds it)"
    GROUPS[tag] = found
    print(f"{tag:10s}", *[str(found[i].parent) for i in sorted(found)][:1], "(5 folds)")
print(f"\nSETUP OK: {sum(len(v) for v in GROUPS.values())} models in {len(GROUPS)} groups [{time.time()-T0:.0f}s]")

input files scanned: 73
code: /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline 
data: /kaggle/input/competitions/rsna-knee-abnormality-detection
v6convnext /kaggle/input/notebooks/theokorir/knee-v6-convnext-5fold/models_v6convnext (5 folds)
v6convnext2 /kaggle/input/notebooks/theokorir/convnext-5-fold-retrain-seed43/models_v6convnext2 (5 folds)

SETUP OK: 10 models in 2 groups [0s]


In [2]:
import torch
ALL = W / "models_all"; shutil.rmtree(ALL, ignore_errors=True); ALL.mkdir()
rows = []
for tag, found in GROUPS.items():
    for i in range(5):
        dst = ALL / f"fold{i}_{tag}.pt"; shutil.copy(found[i], dst)
        ck = torch.load(dst, map_location="cpu")
        rows.append(dict(file=dst.name, backbone=ck["backbone"], K=ck["K"], head=ck.get("head", "series"),
                         canonical_geometry=bool(ck.get("canon", False)), MB=round(dst.stat().st_size / 1e6, 1)))
M = pd.DataFrame(rows); print(M.to_string(index=False))
assert M.K.nunique() == 1, "checkpoints disagree on K: infer.py would silently use the last one"
ANY_CANON = bool(M.canonical_geometry.any())
print(f"\n{len(M)} checkpoints OK | backbones: {sorted(M.backbone.unique())} | token heads: {int((M['head'] == 'tokens').sum())} | canonical-geometry models: {int(M.canonical_geometry.sum())}")

                file                    backbone  K   head  canonical_geometry    MB
 fold0_v6convnext.pt convnext_tiny.in12k_ft_in1k  1 tokens               False 115.5
 fold1_v6convnext.pt convnext_tiny.in12k_ft_in1k  1 tokens               False 115.5
 fold2_v6convnext.pt convnext_tiny.in12k_ft_in1k  1 tokens               False 115.5
 fold3_v6convnext.pt convnext_tiny.in12k_ft_in1k  1 tokens               False 115.5
 fold4_v6convnext.pt convnext_tiny.in12k_ft_in1k  1 tokens               False 115.5
fold0_v6convnext2.pt convnext_tiny.in12k_ft_in1k  1 tokens               False 115.5
fold1_v6convnext2.pt convnext_tiny.in12k_ft_in1k  1 tokens               False 115.5
fold2_v6convnext2.pt convnext_tiny.in12k_ft_in1k  1 tokens               False 115.5
fold3_v6convnext2.pt convnext_tiny.in12k_ft_in1k  1 tokens               False 115.5
fold4_v6convnext2.pt convnext_tiny.in12k_ft_in1k  1 tokens               False 115.5

10 checkpoints OK | backbones: ['convnext_tiny.in12k_ft_in1k'] |

In [3]:
test_ids = pd.read_csv(DATA / "test.csv").StudyInstanceUID
N_TEST = len(test_ids); print("test studies in this run:", N_TEST)
def run_infer(data_dir, out, tag):
    cmd = [sys.executable, str(CODE / "infer.py"), "--models_dir", str(ALL), "--data_dir", str(data_dir), "--out", str(out), "--workers", "4", "--bs", "4"]
    t = time.time(); last = ""; lat = None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(CODE))
    for line in p.stdout:
        last = line.rstrip()
        mm = re.search(r"laterality known for (\d+)/(\d+)", last)
        if mm: lat = (int(mm.group(1)), int(mm.group(2)))
        if "/" in last and "|" in last and last.split("/")[0].isdigit():
            if int(last.split("/")[0]) % 40 == 0: print(f"[{tag}] {last}", flush=True)
        else: print(f"[{tag}] {last}", flush=True)
    rc = p.wait(); wall = time.time() - t
    assert rc == 0, f"infer.py failed (exit {rc})"
    m = re.search(r"\| (\d+)s\s*$", last)
    return wall, (float(m.group(1)) if m else float("nan")), lat
wall_main, loop_main, lat_main = run_infer(DATA, W / "submission.csv", "main")
print(f"\nmain run: wall {wall_main:.0f}s | inference loop {loop_main:.0f}s | load/setup ~{wall_main-loop_main:.0f}s | per study {loop_main/max(N_TEST,1):.2f}s")
if ANY_CANON and N_TEST >= 20 and lat_main and lat_main[0] < 0.9 * lat_main[1]:
    print(f"WARNING: laterality known for only {lat_main[0]}/{lat_main[1]} test studies")

test studies in this run: 3
[main] /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/model.py:64: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
[main]   self.fuse = nn.TransformerEncoder(layer, n_layers)
[main] models: 10 | canonical-geometry models: 0 | token-head models: 10
[main] saved /kaggle/working/submission.csv (3, 13) | unreadable studies: 0 | 16s

main run: wall 38s | inference loop 16s | load/setup ~22s | per study 5.33s


In [4]:
# benchmark only when the test set is tiny (the visible commit run); skipped in the real scoring run
BENCH_N = 100
if N_TEST < 50:
    tr = pd.read_csv(DATA / "train.csv"); ts = pd.read_csv(DATA / "train_series.csv")
    ids = tr.StudyInstanceUID.sample(BENCH_N, random_state=0).tolist()
    B = Path("/tmp/bench"); shutil.rmtree(B, ignore_errors=True); B.mkdir(parents=True)
    pd.DataFrame({"StudyInstanceUID": ids}).to_csv(B / "test.csv", index=False)
    ts[ts.StudyInstanceUID.isin(ids)].to_csv(B / "test_series.csv", index=False)
    cols = list(pd.read_csv(DATA / "sample_submission.csv").columns)
    sb = pd.DataFrame(0.5, index=range(BENCH_N), columns=cols); sb["StudyInstanceUID"] = ids; sb.to_csv(B / "sample_submission.csv", index=False)
    os.symlink(DATA / "train_series", B / "test_series")
    wall_b, loop_b, lat_b = run_infer(B, B / "bench_sub.csv", "bench")
    per = loop_b / BENCH_N; load_s = wall_b - loop_b; proj = load_s + per * N_TEST_FULL_EST
    print(f"\n===== TIMING PROJECTION ({len(M)} models, THIS GPU/CPU) =====")
    print(f"  model load + setup : {load_s:6.0f} s\n  per study          : {per:6.2f} s\n  projected for {N_TEST_FULL_EST} studies: {proj:7.0f} s = {proj/3600:.2f} h (limit 9 h)")
    print(f"  efficiency runtime term RuntimeSeconds/32400 = {proj/32400:.3f}   (+0.01 AUC is worth ~900-1000 s)")
    if proj > 6 * 3600: print("  !! projected over 6 h: too close to the 9 h limit, drop a group")
    if ANY_CANON:
        assert lat_b and lat_b[0] >= 0.9 * lat_b[1], f"laterality resolved for only {lat_b} train studies: do NOT submit canonical-geometry models"
        print(f"  laterality resolved for {lat_b[0]}/{lat_b[1]} benchmark studies")
else:
    print(f"test set has {N_TEST} studies: real scoring run, benchmark skipped.")

[bench] /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/model.py:64: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
[bench]   self.fuse = nn.TransformerEncoder(layer, n_layers)
[bench] models: 10 | canonical-geometry models: 0 | token-head models: 10
[bench] 40/100 | 71s
[bench] 80/100 | 132s
[bench] saved /tmp/bench/bench_sub.csv (100, 13) | unreadable studies: 0 | 163s

===== TIMING PROJECTION (10 models, THIS GPU/CPU) =====
  model load + setup :     21 s
  per study          :   1.63 s
  projected for 1300 studies:    2140 s = 0.59 h (limit 9 h)
  efficiency runtime term RuntimeSeconds/32400 = 0.066   (+0.01 AUC is worth ~900-1000 s)


In [5]:
sub = pd.read_csv(W / "submission.csv"); ref = pd.read_csv(DATA / "sample_submission.csv")
assert list(sub.columns) == list(ref.columns), "columns differ from sample_submission"
assert set(sub.StudyInstanceUID) == set(ref.StudyInstanceUID) and len(sub) == len(ref), "study ids differ"
vals = sub.drop(columns="StudyInstanceUID")
assert vals.notna().all().all() and ((vals >= 0) & (vals <= 1)).all().all(), "NaN or out-of-range values"
print(f"({len(sub)}, {sub.shape[1]}) OK")
try:
    print(f"min over labels of prediction std: {float(vals.std().min()):.3f} (should not be ~0)")
except Exception as e:
    print("std check skipped:", repr(e))
print(f"TOTAL notebook time so far: {time.time()-T0:.0f}s")

(3, 13) OK
min over labels of prediction std: 0.088 (should not be ~0)
TOTAL notebook time so far: 250s
